# Curency COnverter Tool

In [ ]:
api_key="https://v6.exchangerate-api.com/v6/0d5d85558c2c2b1352ce2449/pair/USD/INR"

In [34]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests

In [35]:
# Tool create 

@tool
def get_conversion_factor(base_currency:str, target_currency:str)-> float:
    """ this function fetches the currency factor betwwen a given base currency and a target currency """
    url = f"https://v6.exchangerate-api.com/v6/0d5d85558c2c2b1352ce2449/pair/{base_currency}/{target_currency}"

    response = requests.get(url)
    return response.json()


In [36]:
get_conversion_factor.invoke({
    "base_currency":"USD",
    "target_currency":"INR"   
})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1791936002,
 'time_last_update_utc': 'Wed, 14 Oct 2026 00:00:02 +0000',
 'time_next_update_unix': 1792022402,
 'time_next_update_utc': 'Thu, 15 Oct 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.314}

In [37]:
@tool
def convert(base_currency:int, conversion_rate:Annotated[float,InjectedToolArg])-> float:
    """ give a currency conversion rate this function calculate that target currency value , from a given base value  """
    return base_currency*conversion_rate

In [38]:
convert.invoke({
    'base_currency':10,
    'conversion_rate':85.5
})

855.0

In [39]:
# tool binding 

llm = ChatOllama(
    model="qwen3:4b",
    temperature=0.7
)

llm_with_tools = llm.bind_tools([get_conversion_factor, convert])


messages = [HumanMessage("What is the converter factor between USD and INR , and based on that can you convert 10 USD to INR  ")]

messages


[HumanMessage(content='What is the converter factor between USD and INR , and based on that can you convert 10 USD to INR  ', additional_kwargs={}, response_metadata={})]

In [40]:
result = llm_with_tools.invoke(messages)
messages.append(result)
messages

[HumanMessage(content='What is the converter factor between USD and INR , and based on that can you convert 10 USD to INR  ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-10-14T04:25:26.1222918Z', 'done': True, 'done_reason': 'stop', 'total_duration': 224353642200, 'load_duration': 9216027800, 'prompt_eval_count': 231, 'prompt_eval_duration': 1865939000, 'eval_count': 2225, 'eval_duration': 213249813000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a138a5-0703-73c1-a902-16c8f9d9f84d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': '5ad137be-3830-46d0-a517-dd7a223ff6e0', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency': 10}, 'id': '9b9d04d4-1654-437c-b857-f74e61c1b0a5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 231, 'output

In [41]:
import json
for tool_call in result.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        # print (tool_message1)
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        messages.append(tool_message1)
    if tool_call['name'] =='convert':
        # fetch the current argument 
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

messages



[HumanMessage(content='What is the converter factor between USD and INR , and based on that can you convert 10 USD to INR  ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-10-14T04:25:26.1222918Z', 'done': True, 'done_reason': 'stop', 'total_duration': 224353642200, 'load_duration': 9216027800, 'prompt_eval_count': 231, 'prompt_eval_duration': 1865939000, 'eval_count': 2225, 'eval_duration': 213249813000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a138a5-0703-73c1-a902-16c8f9d9f84d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': '5ad137be-3830-46d0-a517-dd7a223ff6e0', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency': 10, 'conversion_rate': 95.314}, 'id': '9b9d04d4-1654-437c-b857-f74e61c1b0a5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'

In [42]:
res = llm_with_tools.invoke(messages)
res.content

'The currency conversion factor between USD and INR is **95.314**. \n\nUsing this rate, **10 USD** converts to **953.14 INR** (rounded to two decimal places). \n\nThis calculation is based on the latest exchange rate data (as of October 14, 2026).'